### Lasso Regression (L1 Regularization)

Ridge'e benzer şekilde, Lasso da loss fonksiyonuna bir ceza terimi ekler, ama bu sefer katsayıların karesi değil, mutlak değeri kullanılır:

- Loss = Hata + λ × Σ|w|

- λ (alpha) → cezanın gücünü kontrol eden katsayı
- Σ|w| → tüm katsayıların mutlak değerlerinin toplamı (L1 cezası)

### Ridge'den Temel Farkı: Feature Selection

Faz 1'de öğrendiğimiz geometrik yorumdan hatırlarsak, L1 cezasının kısıtlama bölgesi bir elmas şeklindeydi ve bu elmasın köşeleri tam olarak eksenler üzerindeydi. Bu yüzden Lasso, bazı katsayıları tam olarak sıfıra indirebilir yani modelden bir değişkeni tamamen çıkarabilir. Ridge ise katsayıları küçültür ama nadiren tam sıfırlar, tüm değişkenleri (küçük de olsa) modelde tutar.

### Ne Zaman Hangisi Kullanılır?
- Çok sayıda değişken var ve bazılarının gereksiz olduğu düşünülüyorsa → Lasso (gereksizleri otomatik eler, Feature Selection yapar)
- Tüm değişkenlerin az çok faydalı olduğu düşünülüyor, sadece büyüklüklerini kontrol altına almak isteniyorsa → Ridge

### Lambda'nın (alpha) Etkisi
- λ küçük → hafif düzenlileştirme, cezasız modele yakın davranış
- λ arttıkça → daha fazla katsayı sıfıra iner (feature selection güçlenir)
- λ çok büyük → tüm katsayılar sıfırlanabilir, model aşırı basitleşir (underfitting, hatta R² negatife düşebilir)

In [1]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Veri setini yükle ve temizle
df = sns.load_dataset('mpg')
df = df.dropna(subset=['horsepower'])

# 2. Polynomial özellikleri oluştur (degree=3)
poly3 = PolynomialFeatures(degree=3)
X_poly3 = poly3.fit_transform(df[['horsepower']])
y = df['mpg']

# 3. Train/test ayır
X_train, X_test, y_train, y_test = train_test_split(X_poly3, y, test_size=0.2, random_state=42)

# 4. Scaling uygula
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Farklı alpha değerleriyle Lasso dene
for alpha_deger in [0.001, 0.01, 0.1, 1, 10]:
    lasso_model = Lasso(alpha=alpha_deger)
    lasso_model.fit(X_train_scaled, y_train)
    y_pred = lasso_model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    print(f"Alpha={alpha_deger}: R²={r2:.4f}, Katsayılar={lasso_model.coef_}")

Alpha=0.001: R²=0.6400, Katsayılar=[  0.         -18.7153128   13.14430613  -0.44464162]
Alpha=0.01: R²=0.6409, Katsayılar=[  0.         -14.78766692   5.37060895   3.5181238 ]
Alpha=0.1: R²=0.6453, Katsayılar=[  0.         -10.70656944   0.           4.88925209]
Alpha=1: R²=0.5811, Katsayılar=[ 0.         -5.21486731 -0.         -0.        ]
Alpha=10: R²=-0.0114, Katsayılar=[ 0. -0. -0. -0.]


c:\Users\asus\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.498e+03, tolerance: 1.975e+00
  model = cd_fast.enet_coordinate_descent(
c:\Users\asus\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.153e+00, tolerance: 1.975e+00
  model = cd_fast.enet_coordinate_descent(


### Lasso Sonuçları ve Feature Selection Gözlemi

Alpha=0.1'de, x² teriminin katsayısı otomatik olarak sıfıra indi ve model en yüksek R²'yi (0.6453) bu noktada verdi. Bu, Lasso'nun gereksiz bir terimi (x²) modelden çıkararak hem daha basit hem daha başarılı bir model ürettiğinin kanıtı Ridge'deki gözlemimizle (x²'nin az katkı sağladığı) tutarlı.

Alpha arttıkça (1, 10), model giderek daha fazla katsayıyı sıfırladı ve performans düştü; alpha=10'da tüm katsayılar sıfırlanıp R² negatife düştü bu, aşırı düzenlileştirmenin modeli tamamen işlevsiz hale getirdiğini gösteriyor.

Not: İlk katsayının her zaman 0 çıkması Lasso'nun bir kararı değil, PolynomialFeatures'ın eklediği sabit (bias) sütununun scaling sonrası sıfırlanmasından kaynaklanıyor.